In [1]:
import { display } from 'tslab';

# The Zebra Puzzle

The following puzzle appeared in the magazine *Life International* on the 17th of December in the year 1962:
<ol>
    <li>There are five houses.</li>
    <li>The Englishman lives in the red house.</li>
    <li>The Spaniard owns the dog.</li>
    <li>Coffee is drunk in the green house.</li>
    <li>The Ukrainian drinks tea.</li>
    <li>The green house is immediately to the right of the ivory house.</li>
    <li>The Old Gold smoker owns snails.</li>
    <li>Kools are smoked in the yellow house.</li>
    <li>Milk is drunk in the middle house.</li>
    <li>The Norwegian lives in the first house.</li>
    <li>The man who smokes Chesterfields lives in the house next to the man with the fox.</li>
    <li>Kools are smoked in the house next to the house where the horse is kept.</li>
    <li>The Lucky Strike smoker drinks orange juice.</li>
    <li>The Japanese smokes Parliaments.</li>
    <li>The Norwegian lives next to the blue house.</li>
</ol>
Furthermore, each of the five houses is painted in a different colour, their inhabitants are of different nationalities, own different pets, drink different beverages, and smoke different brands of cigarettes.

Your task is to write a program that answers the following questions: 
<ul>
    <li><b>Who drinks water?</b></li>
    <li><b>Who owns the zebra?</b></li>
</ul>

First, we have to import the CSP solver.

In [2]:
import { CSP, Formula, Assignment, solve } from "./02-Backtracking-Constraint-Solver";

In order to succinctly express the constraints that all houses have different colours, the inhabitants have different nationalities etc., it is convenient to implement a function $\texttt{allDifferent}(V)$ that takes a set of variables $V$ and returns a set of formulas that is true if and only if all the variables from $V$ have different values.

In [6]:
function allDifferent(variables: string[]): Formula[] {
    return variables.flatMap(a => variables.filter(b => a < b).map(b => `${a} != ${b}`));
}

In [7]:
allDifferent(['x', 'y', 'z']);

[ 'x != y', 'x != z', 'y != z' ]


In [8]:
function next_to(x: string, y: string): Formula {
    return `${x} - 1 == ${y} || ${x} + 1 == ${y}`;
}

In [9]:
next_to('Chesterfields', 'Fox');

Chesterfields - 1 == Fox || Chesterfields + 1 == Fox


The function $\texttt{zebraCSP}()$ returns a CSP that codes the zebra problem.  When implementing this function it is important to order the variables in a way that variables that are connected to each other by a constraint are tried in succession, for otherwise the CSP solver will take mu$\cdots$uch longer.

In [10]:
const Nations = [ "English", "Spaniard", "Ukrainian", "Norwegian", "Japanese" ];
const Drinks  = [ "Coffee" , "Tea", "Milk", "OrangeJuice", "Water" ];
const Pets    = [ "Dog", "Snails", "Horse", "Fox", "Zebra" ];
const Brands  = [ "LuckyStrike", "Parliaments", "Kools", "Chesterfields", "OldGold" ];
const Colours = [ "Red", "Green", "Ivory", "Yellow", "Blue" ];

In [12]:
function zebraCSP(): CSP { 
    const Constraints: string[] = [];
    Constraints.push(...allDifferent(Nations));
    Constraints.push(...allDifferent(Drinks));
    Constraints.push(...allDifferent(Pets));
    Constraints.push(...allDifferent(Brands));
    Constraints.push(...allDifferent(Colours));
    // There are five houses
    const Values = [1, 2, 3, 4, 5];
    // The Englishman lives in the red house.
    Constraints.push("English == Red");
    // The Spaniard owns the dog.
    Constraints.push("Spaniard == Dog");
    // Coffee is drunk in the green house.
    Constraints.push("Coffee == Green");
    // The Ukrainian drinks tea.
    Constraints.push("Ukrainian == Tea");
    // The green house is immediately to the right of the ivory house.
    Constraints.push("Ivory +1 == Green");
    // The Old Gold smoker owns snails.
    Constraints.push("OldGold == Snails");
    // Kools are smoked in the yellow house.
    Constraints.push("Kools == Yellow");
    // Milk is drunk in the middle house.
    Constraints.push("Milk == 3");
    // The Norwegian lives in the first house.
    Constraints.push("Norwegian == 1");
    // The man who smokes Chesterfields lives in the house next to the man with the fox.
    Constraints.push(next_to("Chesterfields", "Fox"));
    // Kools are smoked in the house next to the house where the horse is kept.
    Constraints.push(next_to("Horse", "Kools"));
    // The Lucky Strike smoker drinks orange juice.
    Constraints.push("LuckyStrike == OrangeJuice");
    // The Japanese smokes Parliaments.
    Constraints.push("Japanese == Parliaments");
    // The Norwegian lives next to the blue house.
    Constraints.push(next_to("Norwegian", "Blue"));
    
    const Variables = [
        ...Nations,
        ...Drinks,
        ...Pets,
        ...Brands,
        ...Colours
    ];
    return [Variables, Values, Constraints];
}

In [13]:
const zebra = zebraCSP();

There are 25 variable and 64 constraints.

In [14]:
[zebra[0].length, zebra[2].length]

[ 25, 64 ]


When the variables are ordered in a sensible way, the problem can be solved in less than a second.  If the variables are ordered randomly, you can expect your computation to take several minutes.

In [15]:
console.time("solution");
const solution = solve(zebra);
console.timeEnd("solution");

solution: 280.803ms


In [16]:
solution

RecursiveMap(25) {
  Blue => 2,
  Chesterfields => 2,
  Coffee => 5,
  Dog => 4,
  English => 3,
  Fox => 1,
  Green => 5,
  Horse => 2,
  Ivory => 4,
  Japanese => 5,
  Kools => 1,
  LuckyStrike => 4,
  Milk => 3,
  Norwegian => 1,
  OldGold => 3,
  OrangeJuice => 4,
  Parliaments => 5,
  Red => 3,
  Snails => 3,
  Spaniard => 4,
  Tea => 2,
  Ukrainian => 2,
  Water => 1,
  Yellow => 1,
  Zebra => 5
}


## Functions to Print the Solution

In [17]:
function showHTML(solution: Assignment) {
    let result = '<table style="border:2px solid blue; text-align:center; font-family:sans-serif;">\n';
    result += '<tr>';
    for (const name of ['House', 'Nationality',  'Drink', 'Animal', 'Brand', 'Colour']) {
        result += `<th style="color:gold; background-color:blue; padding: 5px 10px;">${name}</th>`;
    }
    result += '</tr>\n';
    
    for (let house = 1; house <= 5; house++) {
        result += `<tr><td style="border:1px solid green; padding: 5px;">${house}</td>`;
        for (const category of [Nations, Drinks, Pets, Brands, Colours]) {
            for (const item of category) {
                if (solution.get(item) === house) {
                    result += `<td style="border:1px solid green; padding: 5px;">${item}</td>`;
                }
            }
        }
        result += '</tr>\n';
    }
    result += '</table>';
    display.html(result);
}

In [18]:
if (solution) {
    showHTML(solution);
}

House,Nationality,Drink,Animal,Brand,Colour
1,Norwegian,Water,Fox,Kools,Yellow
2,Ukrainian,Tea,Horse,Chesterfields,Blue
3,English,Milk,Snails,OldGold,Red
4,Spaniard,OrangeJuice,Dog,LuckyStrike,Ivory
5,Japanese,Coffee,Zebra,Parliaments,Green


## Checking the Uniqueness

The function `negateSolution` takes a dictionary that represents a solution and returns a formula
that is true iff one of the variables is different than in the solution.

In [19]:
function negateSolution(solution: Assignment, vars: string[]): string {
    return vars.map(v => `${v} != ${solution.get(v)}`).join(' || ');
}

In [20]:
if (solution) {
    const negatedString = negateSolution(solution, zebra[0]);
    console.log(negatedString.substring(0, 100) + '...'); // Print preview
}

English != 3 || Spaniard != 4 || Ukrainian != 2 || Norwegian != 1 || Japanese != 5 || Coffee != 5 ||...


The function `checkUniqueness` takes two arguments:
* `Solution` is a solution of `CSP`.
* `CSP` is a *constraint satisfaction problem*.

It tries to compute a new solution for the constraint satisfaction problem that is different from the given solution.

In [21]:
function checkUniqueness(solution: Assignment, csp: CSP): Assignment | null {
    const [Vars, Values, Constraints] = csp;
    const newConstraints = [...Constraints, negateSolution(solution, Vars)];
    const newCSP: CSP = [Vars, Values, newConstraints];
    const newSolution = solve(newCSP);
    
    if (newSolution) {
        console.log('The solution is not unique. The alternative solution is:');
        showHTML(newSolution);
        return newSolution;
    } else {
        console.log('Well done! The solution is unique.');
        return null;
    }
}

In [22]:
console.time("uniquenessCheck");
if (solution) {
    checkUniqueness(solution, zebra);
}
console.timeEnd("uniquenessCheck");

Well done! The solution is unique.
uniquenessCheck: 696.579ms
